In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, FloatType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col, sum as _sum, row_number
import numpy as np
from models.fcm import Dfcm
from utils.validity import * 
import time 


# Initialize Spark session
spark = SparkSession.builder.appName("FCM_PySpark").getOrCreate()
spark

# Bước 1: Đọc và chuẩn bị dữ liệu
csv_file_path = "data/csv/602_Dry_Bean.csv"
data = spark.read.csv(csv_file_path, header=True, inferSchema=True)
data = data.drop(data.columns[-1])
for column in data.columns:
    data = data.withColumn(column, col(column).cast(FloatType()))

# ---------Start: Phương pháp Naive Sharding khởi tạo tậm cụm
## Bước 1: Tính tổng các giá trị thuộc tính của một đối tượng và thêm giá trị tổng này thành cột mới vào tập dữ liệu. Thực hiện trên toàn bộ dữ liệu 
data = data.withColumn("sum", sum(col(column) for column in data.columns)) 

## Bước 2: Sắp xếp tập dữ liệu theo cột tổng mới tạo theo thứ tự tăng dần 
data = data.orderBy("sum", ascending=True) 

## Bước 3: Chia tập dữ liệu theo chiều ngang thành k phần bằng nhau, ở đây chia làm 3 dataframe 
window = Window.orderBy("sum")
data_indexed = data.withColumn("row_index", row_number().over(window))

## Define the number of clusters (segments)
k = 7  

## Tính toán số lượng dòng của tập dữ liệu để phân đoạn 
total_rows = data_indexed.count()
segment_size = total_rows // k

## Bước 3: Chia tập dữ liệu theo chiều ngang thành k phần bằng nhau
segments = []
for i in range(k):
    start_idx = i * segment_size + 1
    end_idx = (i + 1) * segment_size
    if i == k - 1:  ## Chắc chắn rằng tất cả các dòng đều được chia hết cho k
        end_idx = total_rows

    segment = data_indexed.filter((col("row_index") >= start_idx) & (col("row_index") <= end_idx)).drop("row_index")
    segments.append(segment)
    
## Bước 4: Đối với mỗi phân đoạn, tính tổng các cột thuộc tính (không bao gồm các cột đã tạo ở bước 1)
## tính giá trị trung bình của nó và đặt vào trong hàng mới. Hàng mới này thực sự là 
## một trong những trọng tâm cụm đã khởi tạo

## Khởi tạo một list rỗng để lưu trữ các trọng tâm cụm   
cluster_centers = []

## Duyệt qua từng phân đoạn
for segment in segments:
    ## Tính tổng các cột thuộc tính 
    segment_sums = segment.agg(*[_sum(col_name).alias(col_name) for col_name in data.columns[:-1]])
    
    ## Tính giá trị trung bình của các cột thuộc tính
    row_count = segment.count()
    cluster_center = [segment_sums.select(col_name).first()[0] / row_count for col_name in data.columns[:-1]]
    
    ## Thêm giá trị trung bình của các cột thuộc tính vào hàng mới
    cluster_centers.append(cluster_center)

print("Initialized Cluster Centers using Naive Sharding:")
for idx, center in enumerate(cluster_centers):
    print(f"Cluster Center {idx + 1}: {center}", len(center))
# ---------End: Phương pháp Naive Sharding khởi tạo tậm cụm

In [2]:
assembler = VectorAssembler(inputCols=data.columns[:-1], outputCol="features")
data = assembler.transform(data)

data_rdd = data.select("features").rdd.map(lambda x: x[0].toArray())    

In [3]:
my_list = []
for i, j in enumerate(np.array(data_rdd.collect())):    
    my_list.append((i, j.tolist()))

In [10]:
# Convert list to RDD
data = my_list
rdd = spark.sparkContext.parallelize(data)

# Broadcast initial cluster centers V to all worker nodes
cluster_centers = spark.sparkContext.broadcast(cluster_centers)

# Step 4: Compute membership matrix U in parallel on each worker node
def compute_membership_matrix(pixel, centers, m=2):
    pixel = np.array(pixel)
    centers = np.array(centers)
    distances = np.linalg.norm(centers - pixel, axis=1)
    distances = np.maximum(distances, 1e-10)  # Prevent division by zero
    u = 1 / (distances ** (2 / (m - 1)))
    u /= np.sum(u)
    return u.tolist()

compute_memberships_udf = udf(lambda features: compute_membership_matrix(features, cluster_centers.value), ArrayType(FloatType()))

# Step 5: Update cluster centers V based on the new membership matrix U
def update_cluster_centers(features, memberships, k):
    features = np.array(features)
    memberships = np.array(memberships)
    new_centers = []

    for i in range(k):
        membership = memberships[:, i]
        numerator = np.sum(membership[:, np.newaxis] * features, axis=0)
        denominator = np.sum(membership)
        new_centers.append(numerator / denominator)

    return np.array(new_centers)

# Convert RDD to DataFrame to use UDFs
df = rdd.toDF(["id", "features"])

# Step 6: Iteratively update V and U until convergence or max iterations
maxLoop = 100
tolerance = 1e-5
t = 0
V_old = spark.sparkContext.broadcast(cluster_centers)
while t < maxLoop:

    print(f"=== Iteration {t + 1} ===")
    
    # Compute memberships for each feature vector
    df = df.withColumn("memberships", compute_memberships_udf(col("features")))

    # Collect the features and memberships for each cluster
    features = np.array(df.select("features").rdd.map(lambda row: row[0]).collect())
    memberships = np.array(df.select("memberships").rdd.map(lambda row: row[0]).collect())

    print("Features Shape:", features.shape)
    print("Memberships Shape:", memberships.shape)

    # Update cluster centers V
    V_new = update_cluster_centers(features, memberships, k)

    # Check for convergence
    diff = np.linalg.norm(V_new - V_old.value)
    print(f"Cluster Centers Difference: {diff}")

    if diff < tolerance:
        print("Convergence reached!")
        break

    # Update broadcast variable
    V_old.unpersist()
    V_old = spark.sparkContext.broadcast(V_new)
    
    t += 1

# Step 7: Output the final membership matrix U and cluster centers V
final_U = df.select("id", "memberships").collect()
final_V = cluster_centers.value

print("Final Membership Matrix U:")
for row in final_U:
    print(row)

print("\nFinal Cluster Centers V:")
print(final_V)

NameError: name 'cluster_centers' is not defined

In [ ]:
# Stop Spark session
spark.stop()